# Benchmarking comparison

Compare regional framework-aligned RMSDs for Our Model, ABB4, ABB3, and Boltz on the benchmark test set (benchmarking_meta.csv).

This code produced Figure 3.3

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

BENCHMARK_DIR = Path("/opig-shared/users/lina4783/abb4_experiments/evaluation/benchmarking")
BENCHMARK_META = Path("/opig-shared/users/lina4783/structures_final/benchmarking_meta.csv")
FILTERED_DIR = BENCHMARK_DIR / "filtered_metrics"
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "Our Model": Path(
        "/opig-shared/users/lina4783/abb4_experiments/evaluation/"
        "predictions_ckpt_5139_imgt/struc_pred_metrics_test_summary.csv"
    ),
    "ABB4": Path(
        "/opig-shared/users/lina4783/abb4_experiments/evaluation/"
        "predictions_abb4_nontrain_imgt/struc_pred_metrics_test_summary.csv"
    ),
    "ABB3": Path(
        "/opig-shared/users/lina4783/abodybuilder3/"
        "predictions_nonoverlap_test_plddt_imgt/struc_pred_metrics_test.csv"
    ),
    "Boltz": Path(
        "/opig-shared/users/lina4783/boltz/"
        "predictions_nonoverlap_test_boltz_imgt/struc_pred_metrics_test.csv"
    ),
}

MODEL_COLORS = {
    "Our Model": "#E8A0BF",  # dusty rose
    "ABB4": "#B8A9D4",       # soft lilac
    "ABB3": "#A8D5C7",       # sage mint
    "Boltz": "#F4B8C5",       # blush pink
}

METRIC_COLS = [
    "H_cdr_all", "H_framework", "H_cdr1", "H_cdr2", "H_cdr3",
    "L_cdr_all", "L_framework", "L_cdr1", "L_cdr2", "L_cdr3",
]
REGION_LABELS = [
    "H_CDRs", "fwH", "CDRH1", "CDRH2", "CDRH3",
    "L_CDRs", "fwL", "CDRL1", "CDRL2", "CDRL3",
]

N_BOOT = 2000
RANDOM_SEED = 0

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"font.size": 11, "axes.titlesize": 13, "axes.labelsize": 12})

In [ ]:
def bootstrap_mean_ci(values, n_boot: int = N_BOOT, seed: int = RANDOM_SEED, ci=(2.5, 97.5)):
    arr = np.asarray(values, dtype=float)
    arr = arr[~np.isnan(arr)]
    if arr.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        boot_means[i] = rng.choice(arr, size=arr.size, replace=True).mean()
    mean = arr.mean()
    low, high = np.percentile(boot_means, ci)
    return mean, low, high

In [ ]:
benchmark_meta = pd.read_csv(BENCHMARK_META)
benchmark_ids = set(benchmark_meta["pdb_name"])
print(f"Benchmark structures: {len(benchmark_ids)}")

coverage_rows = []
filtered_by_model = {}

for name, path in MODEL_PATHS.items():
    raw = pd.read_csv(path)
    in_benchmark = raw["pdb_name"].isin(benchmark_ids)
    found = raw.loc[in_benchmark]
    ok = found.loc[found["status"] == "ok"].copy()
    filtered_by_model[name] = ok
    ok.to_csv(FILTERED_DIR / f"{name.replace(' ', '_').lower()}_benchmark_ok.csv", index=False)
    coverage_rows.append(
        {
            "model": name,
            "in_benchmark_meta": len(benchmark_ids),
            "found_in_metrics": len(found),
            "status_ok": len(ok),
            "missing_from_metrics": len(benchmark_ids - set(raw["pdb_name"])),
            "failed_status": int((found["status"] != "ok").sum()),
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
#coverage_df.to_csv(BENCHMARK_DIR / "coverage_summary.csv", index=False)
coverage_df

In [ ]:
paired_ids = set.intersection(
    *(set(df["pdb_name"]) for df in filtered_by_model.values())
)
paired_ids = sorted(paired_ids)
print(f"Paired structures (ok in all 4 models): {len(paired_ids)}")

missing_from_any = sorted(benchmark_ids - set(paired_ids))
print(f"Benchmark structures not in paired set: {len(missing_from_any)}")

(BENCHMARK_DIR / "paired_pdb_names.txt").write_text("\n".join(paired_ids) + "\n")

paired_by_model = {
    name: df[df["pdb_name"].isin(paired_ids)].copy()
    for name, df in filtered_by_model.items()
}

In [ ]:
summary_rows = []
plot_data = {}

for model_name, df in paired_by_model.items():
    means, lowers, uppers = [], [], []
    for col in METRIC_COLS:
        mean, low, high = bootstrap_mean_ci(df[col].values)
        means.append(mean)
        lowers.append(mean - low)
        uppers.append(high - mean)
        summary_rows.append(
            {
                "model": model_name,
                "metric": col,
                "region": REGION_LABELS[METRIC_COLS.index(col)],
                "n": len(df),
                "mean": mean,
                "ci_low": low,
                "ci_high": high,
            }
        )
    plot_data[model_name] = {
        "means": np.array(means),
        "yerr": np.array([lowers, uppers]),
    }

bootstrap_summary = pd.DataFrame(summary_rows)
#bootstrap_summary.to_csv(BENCHMARK_DIR / "bootstrap_summary.csv", index=False)
bootstrap_summary.head(12)

In [ ]:
x = np.arange(len(REGION_LABELS))
width = 0.20
offsets = [-1.5, -0.5, 0.5, 1.5]
paired_n = len(paired_ids)

fig, ax = plt.subplots(figsize=(13, 5.5))

all_tops = []
for offset, model_name in zip(offsets, MODEL_PATHS.keys()):
    data = plot_data[model_name]
    bars = ax.bar(
        x + offset * width,
        data["means"],
        width,
        yerr=data["yerr"],
        capsize=3,
        label=model_name,
        color=MODEL_COLORS[model_name],
        edgecolor="black",
        linewidth=0.5,
        alpha=0.95,
        error_kw={"elinewidth": 1, "capthick": 1, "ecolor": "black"},
    )
    all_tops.extend((data["means"] + data["yerr"][1]).tolist())
    for bar, val, err in zip(bars, data["means"], data["yerr"][1]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + err + 0.03,
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=6.5,
            rotation=90,
            color="black",
        )

ax.set_ylabel("Mean RMSD (Å)")
ax.set_xlabel("Region")
ax.set_title(f"Framework-aligned RMSD by region (benchmark set, N={paired_n})")
ax.set_xticks(x)
ax.set_xticklabels(REGION_LABELS)
ax.set_ylim(0, max(all_tops) * 1.18)
ax.legend(title="Model", frameon=True)
sns.despine(ax=ax)
fig.tight_layout()

fig.savefig(BENCHMARK_DIR / "rmsd_comparison_benchmark_5139.png", dpi=300, bbox_inches="tight")
fig.savefig(BENCHMARK_DIR / "rmsd_comparison_benchmark_5139.pdf", bbox_inches="tight")
plt.show()

In [ ]:
box_rows = []
for model_name, df in paired_by_model.items():
    for col, region in zip(METRIC_COLS, REGION_LABELS):
        for val in df[col].dropna():
            box_rows.append({"Model": model_name, "Region": region, "RMSD": val})

box_plot_df = pd.DataFrame(box_rows)

fig, ax = plt.subplots(figsize=(13, 5.5))
sns.boxplot(
    data=box_plot_df,
    x="Region",
    y="RMSD",
    hue="Model",
    order=REGION_LABELS,
    hue_order=list(MODEL_PATHS.keys()),
    palette=MODEL_COLORS,
    width=0.75,
    showfliers=False,
    linewidth=0.8,
    linecolor="black",
    ax=ax,
)

for patch in ax.patches:
    patch.set_alpha(0.95)

ax.set_ylabel("Framework-aligned RMSD (Å)")
ax.set_xlabel("Region")
ax.set_title(f"RMSD distributions by region (benchmark set, N={len(paired_ids)})")
ax.legend(title="Model", frameon=True)
sns.despine(ax=ax)
fig.tight_layout()

fig.savefig(BENCHMARK_DIR / "rmsd_boxplot_benchmark_5139.png", dpi=300, bbox_inches="tight")
fig.savefig(BENCHMARK_DIR / "rmsd_boxplot_benchmark_5139.pdf", bbox_inches="tight")
plt.show()

In [ ]:
for name,path in MODEL_PATHS.items():
    df=pd.read_csv(path)
    in_benchmark = df["pdb_name"].isin(benchmark_ids)
    found = df.loc[in_benchmark]
    cdrh3_mean=found["H_cdr3"].mean()
    cdrl3_mean=found["L_cdr3"].mean()
    print(f"{name} CDRH3 mean: {cdrh3_mean:.2f}, CDRL3 mean: {cdrl3_mean:.2f}")


In [ ]:
coverage_df.head()